# Fine-tune Whisper on Nepali — free Colab/Kaggle T4

**Self-contained.** Downloads OpenSLR SLR54 (Google's Large Nepali ASR, CC BY-SA 4.0),
builds the `{audio, text}` dataset, fine-tunes `whisper-small`, and reports WER vs the base
model. No repo, no keys, no local setup.

### Before you run
1. **Colab:** Runtime -> Change runtime type -> **T4 GPU**. (Kaggle: Settings -> Accelerator -> **GPU T4 x2**.)
2. Run cells top to bottom (Runtime -> Run all).

A first pass (`N_TRAIN=3000`, 3 epochs) finishes in roughly 1-2 h on a free T4 and should
already beat base Whisper on Nepali. Raise `N_TRAIN` / `PARTS` for a stronger model.

> Note: OpenSLR is read Nepali, not **Ninglish** (Nepali-English code-switch). Public data
> fixes base Nepali; your in-app correction loop is what wins Ninglish. Layer corrections on
> top of this checkpoint later.


## 1. Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU! Set Runtime -> T4 GPU and re-run.'


## 2. Install dependencies


In [ ]:
!pip -q install -U 'transformers>=4.46' 'datasets>=2.20' 'accelerate>=0.33' jiwer soundfile librosa
print('deps installed')


## 3. Config — tune these for time vs quality


In [ ]:
# Which SLR54 parts to pull (0..f). Each part ~580 MB / ~10k clips.
PARTS      = ['0']            # e.g. ['0','1','2'] for more data
BASE       = 'openai/whisper-small'   # 'whisper-tiny' = faster test; 'whisper-medium' if time allows
LANGUAGE   = 'nepali'
N_TRAIN    = 3000            # cap training clips (None = use all downloaded)
N_TEST     = 300             # held-out clips for honest WER
EPOCHS     = 3
BATCH      = 16             # T4 handles 16 for whisper-small w/ fp16 + grad checkpointing
OUT_DIR    = 'whisper-ne'
print('config set')


## 4. Download + unzip the data (curl, no auth)


In [ ]:
import os, subprocess, pathlib
DATA = pathlib.Path('slr54'); DATA.mkdir(exist_ok=True)
BASE_URL = 'https://www.openslr.org/resources/54'
# transcript index (FileID <tab> UserID <tab> Devanagari text)
if not (DATA/'utt_spk_text.tsv').exists():
    !curl -L -o slr54/utt_spk_text.tsv {BASE_URL}/utt_spk_text.tsv
for p in PARTS:
    z = DATA/f'asr_nepali_{p}.zip'
    if not z.exists():
        !curl -L -o {str(z)} {BASE_URL}/asr_nepali_{p}.zip
    !cd slr54 && unzip -q -o asr_nepali_{p}.zip
n_flac = len(list(DATA.rglob('*.flac')))
print('flac files extracted:', n_flac)


## 5. Build train / test JSONL (join TSV to audio)


In [ ]:
import json, random
audio = {p.stem: str(p) for p in DATA.rglob('*.flac')}
rows = []
for line in open(DATA/'utt_spk_text.tsv', encoding='utf-8'):
    parts = line.rstrip('\n').split('\t')
    if len(parts) < 3: continue
    fid, text = parts[0], '\t'.join(parts[2:]).strip()
    if text and fid in audio:
        rows.append({'audio': audio[fid], 'text': text})
random.seed(42); random.shuffle(rows)
test  = rows[:N_TEST]
train = rows[N_TEST: N_TEST + N_TRAIN] if N_TRAIN else rows[N_TEST:]
json.dump(train, open('train.json','w'), ensure_ascii=False)
json.dump(test,  open('test.json','w'),  ensure_ascii=False)
print(f'train={len(train)}  test={len(test)}')
print('sample:', train[0]['text'])


## 6. Load processor + datasets, extract features


In [ ]:
from datasets import Dataset, Audio
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE, language=LANGUAGE, task='transcribe')

def to_ds(path):
    import json
    ds = Dataset.from_list(json.load(open(path)))
    return ds.cast_column('audio', Audio(sampling_rate=16000))

train_ds, test_ds = to_ds('train.json'), to_ds('test.json')

def prepare(b):
    a = b['audio']
    b['input_features'] = processor.feature_extractor(a['array'], sampling_rate=16000).input_features[0]
    b['labels'] = processor.tokenizer(b['text']).input_ids
    return b

# Single-process map: num_proc>1 crashes on Colab decoding audio ("subprocess died").
train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names)
test_ds  = test_ds.map(prepare,  remove_columns=test_ds.column_names)
print('features ready')


## 7. Collator + WER metric


In [ ]:
import torch, jiwer
from dataclasses import dataclass
from typing import Any

@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        inp = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(inp, return_tensors='pt')
        lab = self.processor.tokenizer.pad([{'input_ids': f['labels']} for f in features], return_tensors='pt')
        labels = lab['input_ids'].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

collator = Collator(processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': 100 * jiwer.wer(label_str, pred_str)}


## 8. Train (free T4)


In [ ]:
from transformers import (WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments)

model = WhisperForConditionalGeneration.from_pretrained(BASE)
model.generation_config.language = LANGUAGE
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False   # required with gradient_checkpointing

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    fp16=True,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=25,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,   # transformers v5 renamed tokenizer= -> processing_class=
)
trainer.train()
trainer.save_model(OUT_DIR); processor.save_pretrained(OUT_DIR)
print('saved ->', OUT_DIR)


## 9. Did it beat base? (WER: base vs fine-tuned)


In [ ]:
import torch, jiwer, json
from transformers import pipeline
test_rows = json.load(open('test.json'))
refs   = [r['text'] for r in test_rows]
audios = [r['audio'] for r in test_rows]

def wer_of(model_path):
    asr = pipeline('automatic-speech-recognition', model=model_path, device=0,
                   generate_kwargs={'language': LANGUAGE, 'task': 'transcribe'})
    hyps = [asr(a)['text'] for a in audios]
    return 100 * jiwer.wer(refs, hyps)

base_wer = wer_of(BASE)
ft_wer   = wer_of(OUT_DIR)
print(f'{BASE:30} WER = {base_wer:.2f}')
print(f'{OUT_DIR:30} WER = {ft_wer:.2f}')
print('IMPROVED' if ft_wer < base_wer else 'no gain - need more/cleaner data')


## 10. Keep the model

Download the folder, or push to the Hugging Face Hub (free) to serve later as a worker STT
provider (`STT_PROVIDER=finetuned`, see `docs/FINETUNE.md`).


In [ ]:
# Option A: zip + download
!zip -qr whisper-ne.zip {OUT_DIR}
try:
    from google.colab import files; files.download('whisper-ne.zip')
except Exception:
    print('Not on Colab - find whisper-ne.zip in the file browser.')

# Option B: push to the HF Hub (uncomment; needs a free token from hf.co/settings/tokens)
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub('your-username/whisper-small-ne')
# processor.push_to_hub('your-username/whisper-small-ne')
